## <code>static Word Embedding</code>

In [1]:
import pandas as pd
import numpy as np
import string
import re
import math
import os
import gensim
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from typing import Dict, List, Tuple
from collections import defaultdict

### Word Translation for Testing 

In [ ]:
class WordTranslator:
    def __init__(self, word_embeddings_path: str):
        """
        Initialize the word translator with pre-trained word embeddings
        """
        self.word_embeddings = gensim.models.KeyedVectors.load_word2vec_format(
            word_embeddings_path, binary=True
        )
        self.vocabulary = set(self.word_embeddings.index_to_key)
        
    def get_nearest_neighbors(self, word: str, k: int = 8) -> List[Tuple[str, float]]:
        """
        Find k nearest neighbors for a given word based on cosine similarity
        """
        if word not in self.vocabulary:
            return []
        
        similar_words = self.word_embeddings.most_similar(word, topn=k)
        return similar_words
    
    def calculate_translation_probability(self, source_word: str, target_word: str,
                                       temperature: float = 0.1) -> float:
        """
        Calculate translation probability pt(w|u) using NTLM approach
        Args:
            source_word: The source word u
            target_word: The target word w
            temperature: Temperature parameter for controlling probability distribution
        """
        if source_word not in self.vocabulary or target_word not in self.vocabulary:
            return 0.0
        
        # Calculate cosine similarity
        similarity = self.word_embeddings.similarity(source_word, target_word)
        
        # Convert similarity to probability using softmax-like normalization
        # Higher temperature leads to more uniform probabilities
        probability = math.exp(similarity / temperature)
        
        # Get normalization term (sum of exp(sim(u,w')/temperature) for all w' in vocabulary)
        # Note: For efficiency, we only consider top-k nearest neighbors
        nearest_neighbors = self.get_nearest_neighbors(source_word, k=8)
        normalization_term = sum(math.exp(sim / temperature) 
                               for _, sim in nearest_neighbors)
        
        return probability / normalization_term if normalization_term > 0 else 0.0
    
    def get_translation_candidates(self, source_word: str, 
                                 threshold: float = 0.01) -> List[Tuple[str, float]]:
        """
        Get all possible translation candidates with their probabilities
        Args:
            source_word: The source word to translate
            threshold: Minimum probability threshold for considering a translation
        """
        if source_word not in self.vocabulary:
            return []
        
        # Get nearest neighbors as potential translation candidates
        candidates = self.get_nearest_neighbors(source_word, k=8)
        
        # Calculate translation probabilities for each candidate
        translation_probs = []
        for candidate, _ in candidates:
            prob = self.calculate_translation_probability(source_word, candidate)
            if prob >= threshold:
                translation_probs.append((candidate, prob))
        
        # Sort by probability in descending order
        return sorted(translation_probs, key=lambda x: x[1], reverse=True)

### Preprocessing for WSJ datasets

In [2]:
# WSJ Data

def parse_trec_file(trec_file_path):
    doc_texts = {}
    current_doc_id = None
    current_text = []
    
    encodings = ['utf-8', 'latin-1', 'ISO-8859-1']
    for encoding in encodings:
        try:
            with open(trec_file_path, 'r', encoding=encoding, errors='ignore') as file:
                for line in file:
                    if line.startswith('<DOCNO>'):
                        current_doc_id = line.strip().replace('<DOCNO>', '').replace('</DOCNO>', '').strip()
                    elif line.startswith('</TEXT>'):
                        if current_doc_id:
                            doc_texts[current_doc_id] = ' '.join(current_text)
                            current_doc_id = None
                            current_text = []
                    elif current_doc_id:
                        if not (line.startswith('<DOC>') or line.startswith('</DOC>') or line.startswith('<FILEID>') or
                                line.startswith('<FIRST>') or line.startswith('<SECOND>') or line.startswith('<HEAD>') or
                                line.startswith('<DATELINE>') or line.startswith('<TEXT>') or 
                                line.startswith('<HL>') or line.startswith('</HL>') or 
                                line.startswith('<DD>') or line.startswith('</DD>') or 
                                line.startswith('<SO>') or line.startswith('</SO>') or 
                                line.startswith('<IN>') or line.startswith('</IN>')):
                            current_text.append(line.strip())
            break
        except UnicodeDecodeError:
            continue  

    return doc_texts

# Path to your concatenated TREC file
#trec_file_path = os.path.join("..", "Data", "WSJ_DOC", "wsj", "concatenated_WSJ", "concatenated_WSJ.txt")
trec_file_path = r"C:\Eyasu\Thesis_files\DOTGOV\DOTGOV\Full DOTGOV_concatenated\cleaned_concatenated_files.txt"

# Parse the document texts
doc_texts = parse_trec_file(trec_file_path)

In [3]:
dict(list(doc_texts.items())[0:5])

{}

### Skipgram Model For WSJ Datasets

In [ ]:
class SkipgramTrainer:
    def __init__(self):
        """Initialize the trainer with necessary NLTK resources"""
        nltk.download('punkt')
        nltk.download('stopwords')
        self.stop_words = set(stopwords.words('english'))
        
    def preprocess_text(self, text: str) -> List[str]:
        """
        Preprocess text by removing special characters, converting to lowercase,
        removing stopwords and tokenizing
        """
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords and non-alphabetic tokens
        tokens = [token for token in tokens 
                 if token not in self.stop_words 
                 and token.isalpha()]
        
        return tokens

    def prepare_training_data(self, doc_texts: Dict[str, str]) -> List[List[str]]:
        """Convert document dictionary into format suitable for Word2Vec training"""
        training_data = []
        
        for doc_id, text in doc_texts.items():
            tokens = self.preprocess_text(text)
            if tokens:  # Only add if document contains valid tokens
                training_data.append(tokens)
                
        return training_data

    def train_skipgram_model(self, 
                            training_data: List[List[str]], 
                            vector_size: int = 300,
                            window: int = 5,
                            min_count: int = 5,
                            workers: int = 4,
                            epochs: int = 10) -> Word2Vec:
        """
        Train Skip-gram model using preprocessed data
        
        Args:
            training_data: List of tokenized documents
            vector_size: Dimensionality of word vectors
            window: Maximum distance between current and predicted word
            min_count: Minimum frequency of words to consider
            workers: Number of CPU cores to use
            epochs: Number of training epochs
        """
        model = Word2Vec(sentences=training_data,
                        vector_size=vector_size,
                        window=window,
                        min_count=min_count,
                        workers=workers,
                        sg=1,  # Skip-gram model (sg=1)
                        epochs=epochs)
        
        return model

    def save_model(self, model: Word2Vec, save_path: str):
        """Save the trained model in word2vec binary format"""
        model.wv.save_word2vec_format(save_path, binary=True)

def main():
    # Initialize trainer
    trainer = SkipgramTrainer()
    
    # Parse TREC file (using your existing parse_trec_file function)
    #trec_file_path = os.path.join("..", "Data", "AP_Doc", "ap", "concatenated", "concatenated_documents.txt")
    #trec_file_path = os.path.join("..", "Data", "WSJ_DOC", "wsj", "concatenated_WSJ", "concatenated_WSJ.txt")
    trec_file_path = r"C:\Eyasu\Thesis_files\DOTGOV\DOTGOV\Full DOTGOV_concatenated\concatenated_files.txt"



    doc_texts = parse_trec_file(trec_file_path)
    
    # Prepare training data
    print("Preparing training data...")
    training_data = trainer.prepare_training_data(doc_texts)
    
    # Train model
    print("Training Skip-gram model...")
    model = trainer.train_skipgram_model(training_data)
    
    # Save model
    save_path = os.path.join("..", "Data", "Word_Embedding", "DOTGOV_skipgram_model10.bin")
    print(f"Saving model to {save_path}...")
    trainer.save_model(model, save_path)
    
    # Test the model with your existing WordTranslator
    print("\nTesting translation capabilities...")
    translator = WordTranslator(save_path)
    
    test_words = ["oil", "gas", "energy", "company"]
    for word in test_words:
        print(f"\nTranslation candidates for '{word}':")
        translations = translator.get_translation_candidates(word)
        for target_word, prob in translations:
            print(f"{target_word}: {prob:.4f}")

if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dolla\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dolla\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preparing training data...
Training Skip-gram model...
Saving model to ..\Data\Word_Embedding\AP_skipgram_model35.bin...

Testing translation capabilities...

Translation candidates for 'oil':
crude: 0.5012
petroleum: 0.1484
barrels: 0.0753
spill: 0.0579
barrel: 0.0571
heating: 0.0557
refineries: 0.0527
opec: 0.0518

Translation candidates for 'gas':
tear: 0.2549
natural: 0.1652
canisters: 0.1222
gasoline: 0.0980
isocynate: 0.0955
propanebutane: 0.0918
nonproducing: 0.0873
methane: 0.0851

Translation candidates for 'energy':
ahearne: 0.1660
nonfossil: 0.1556
energys: 0.1388
fuels: 0.1164
commerce: 0.1111
redoglio: 0.1101
watkins: 0.1037
tokamak: 0.0983

Translation candidates for 'company':
companys: 0.3569
companies: 0.1554
subsidiary: 0.1493
subsidiaries: 0.0823
maker: 0.0702
manufacturer: 0.0648
corporation: 0.0629
anac: 0.0581


### CBOW Model For WSJ

In [ ]:
class CBOWTrainer:
    def __init__(self):
        """Initialize the trainer with necessary NLTK resources"""
        nltk.download('punkt', quiet=True)
        nltk.download('stopwords', quiet=True)
        self.stop_words = set(stopwords.words('english'))
        
    def preprocess_text(self, text: str) -> List[str]:
        """
        Preprocess text by removing special characters, converting to lowercase,
        removing stopwords and tokenizing
        """
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords, non-alphabetic tokens, and very short words
        tokens = [token for token in tokens 
                 if token not in self.stop_words 
                 and token.isalpha()
                 and len(token) > 2]  # Remove very short words
        
        return tokens

    def prepare_training_data(self, doc_texts: Dict[str, str]) -> List[List[str]]:
        """Convert document dictionary into format suitable for Word2Vec training"""
        training_data = []
        total_tokens = 0
        
        for doc_id, text in doc_texts.items():
            tokens = self.preprocess_text(text)
            if tokens:
                training_data.append(tokens)
                total_tokens += len(tokens)
        
        #print(f"Prepared {len(training_data)} documents with {total_tokens} total tokens")
        return training_data

    def train_cbow_model(self, 
                        training_data: List[List[str]], 
                        vector_size: int = 300,
                        window: int = 5,
                        min_count: int = 5,
                        workers: int = 4,
                        epochs: int = 30,
                        negative: int = 15,
                        alpha: float = 0.025,
                        min_alpha: float = 0.0001) -> Word2Vec:
        """
        Train CBOW model using preprocessed data with improved parameters
        
        Args:
            training_data: List of tokenized documents
            vector_size: Dimensionality of word vectors
            window: Maximum distance between current and predicted word
            min_count: Minimum frequency of words to consider
            workers: Number of CPU cores to use
            epochs: Number of training epochs
            negative: Number of negative samples
            alpha: Initial learning rate
            min_alpha: Minimum learning rate
        """
        # Calculate dynamic learning rate decay
        alpha_delta = (alpha - min_alpha) / epochs
        
        # Initialize model with improved parameters
        model = Word2Vec(vector_size=vector_size,
                        window=window,
                        min_count=min_count,
                        workers=workers,
                        sg=0,  # CBOW model
                        negative=negative,
                        alpha=alpha,
                        min_alpha=min_alpha,
                        compute_loss=True)
        
        # Build vocabulary
        model.build_vocab(training_data)
        
        # Train the model with progress monitoring
        total_examples = len(training_data)
        
        losses = []
        for epoch in range(epochs):
            current_alpha = alpha - (alpha_delta * epoch)
            model.alpha = current_alpha
            model.min_alpha = current_alpha
            
            model.train(training_data,
                       total_examples=total_examples,
                       epochs=1,
                       compute_loss=True)
            
            current_loss = model.get_latest_training_loss()
            losses.append(current_loss)
        
        return model

    def save_model(self, model: Word2Vec, save_path: str):
        """Save the trained model in word2vec binary format"""
        model.wv.save_word2vec_format(save_path, binary=True)
        
    
    def evaluate_model(self, model: Word2Vec, test_words: List[str]):
        """
        Evaluate the model by printing similar words and their similarities
        for a list of test words
        """
        print("\nModel Evaluation:")
        for word in test_words:
            try:
                similar_words = model.wv.most_similar(word, topn=5)
                print(f"\nSimilar words to '{word}':")
                for similar_word, similarity in similar_words:
                    print(f"  {similar_word}: {similarity:.4f}")
            except KeyError:
                print(f"\nWord '{word}' not in vocabulary")


def main():
    # Initialize trainer
    trainer = CBOWTrainer()
    
    # Parse TREC file (using your existing parse_trec_file function)
    #trec_file_path = os.path.join("..", "Data", "AP_Doc", "ap", "concatenated", "concatenated_documents.txt")
    trec_file_path = os.path.join("..", "Data", "WSJ_DOC", "wsj", "concatenated_WSJ", "concatenated_WSJ.txt")

    doc_texts = parse_trec_file(trec_file_path)
    
    # Prepare training data
    print("Preparing training data...")
    training_data = trainer.prepare_training_data(doc_texts)
    
    # Train model with the following parameters
    print("Training CBOW model...")
    model = trainer.train_cbow_model(
        training_data,
        vector_size=300,
        window=5,
        min_count=5,
        workers=4,
        epochs=30,
        negative=15,
        alpha=0.025,
        min_alpha=0.0001
    )
    
    # Save model
    save_path = os.path.join("..", "Data", "Word_Embedding", "WSJ_cbow_model.bin")
    print(f"Saving model to {save_path}...")
    trainer.save_model(model, save_path)
    
    # Evaluate model
    test_words = ["oil", "gas", "energy", "company", "pneumonia"]
    trainer.evaluate_model(model, test_words)

if __name__ == "__main__":
    main()

Preparing training data...
Training CBOW model...
Saving model to ..\Data\Word_Embedding\WSJ_cbow_model.bin...

Model Evaluation:

Similar words to 'oil':
  crudeoil: 0.5901
  petroleum: 0.5393
  wellhead: 0.5278
  oils: 0.5068
  naturalgas: 0.4861

Similar words to 'gas':
  naturalgas: 0.7154
  gass: 0.4934
  pipeline: 0.4856
  wainoco: 0.4742
  gasrelated: 0.4730

Similar words to 'energy':
  energys: 0.5834
  transportation: 0.4638
  oil: 0.4558
  naturalgas: 0.4370
  gas: 0.4284

Similar words to 'company':
  companys: 0.7241
  concern: 0.6423
  companies: 0.5624
  maker: 0.4718
  retailer: 0.4680

Word 'pneumoni' not in vocabulary


In [1]:
import os
import re
import string
from typing import Dict, List, Generator
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

def parse_trec_file_in_chunks(trec_file_path: str, chunk_size: int = 1024 * 1024) -> Dict[str, str]:
    """
    Parse a large TREC file in chunks to avoid memory errors.
    """
    doc_texts = {}
    current_doc_id = None
    current_text = []
    
    encodings = ['utf-8', 'latin-1', 'ISO-8859-1']
    for encoding in encodings:
        try:
            with open(trec_file_path, 'r', encoding=encoding, errors='ignore') as file:
                while True:
                    chunk = file.readlines(chunk_size)
                    if not chunk:
                        break
                    for line in chunk:
                        if line.startswith('<DOCNO>'):
                            current_doc_id = line.strip().replace('<DOCNO>', '').replace('</DOCNO>', '').strip()
                        elif line.startswith('</TEXT>'):
                            if current_doc_id:
                                doc_texts[current_doc_id] = ' '.join(current_text)
                                current_doc_id = None
                                current_text = []
                        elif current_doc_id:
                            if not (line.startswith('<DOC>') or line.startswith('</DOC>') or line.startswith('<FILEID>') or
                                    line.startswith('<FIRST>') or line.startswith('<SECOND>') or line.startswith('<HEAD>') or
                                    line.startswith('<DATELINE>') or line.startswith('<TEXT>') or 
                                    line.startswith('<HL>') or line.startswith('</HL>') or 
                                    line.startswith('<DD>') or line.startswith('</DD>') or 
                                    line.startswith('<SO>') or line.startswith('</SO>') or 
                                    line.startswith('<IN>') or line.startswith('</IN>')):
                                current_text.append(line.strip())
            break
        except UnicodeDecodeError:
            continue  

    return doc_texts

class SkipgramTrainer:
    def __init__(self):
        """Initialize the trainer with necessary NLTK resources"""
        self.stop_words = set(stopwords.words('english'))
        
    def preprocess_text(self, text: str) -> List[str]:
        """
        Preprocess text by removing special characters, converting to lowercase,
        removing stopwords, and tokenizing.
        """
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)
        
        # Convert to lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords and non-alphabetic tokens
        tokens = [token for token in tokens 
                 if token not in self.stop_words 
                 and token.isalpha()]
        
        return tokens

    def prepare_training_data(self, doc_texts: Dict[str, str]) -> Generator[List[str], None, None]:
        """Convert document dictionary into format suitable for Word2Vec training"""
        for doc_id, text in doc_texts.items():
            tokens = self.preprocess_text(text)
            if tokens:  # Only yield if document contains valid tokens
                yield tokens

    def train_skipgram_model(self, 
                            training_data: Generator[List[str], None, None], 
                            vector_size: int = 300,
                            window: int = 5,
                            min_count: int = 5,
                            workers: int = 4,
                            epochs: int = 10) -> Word2Vec:
        """
        Train Skip-gram model using preprocessed data.
        
        Args:
            training_data: Generator of tokenized documents
            vector_size: Dimensionality of word vectors
            window: Maximum distance between current and predicted word
            min_count: Minimum frequency of words to consider
            workers: Number of CPU cores to use
            epochs: Number of training epochs
        """
        model = Word2Vec(sentences=training_data,
                        vector_size=vector_size,
                        window=window,
                        min_count=min_count,
                        workers=workers,
                        sg=1,  # Skip-gram model (sg=1)
                        epochs=epochs)
        
        return model

    def save_model(self, model: Word2Vec, save_path: str):
        """Save the trained model in word2vec binary format"""
        model.wv.save_word2vec_format(save_path, binary=True)

def main():
    # Initialize trainer
    trainer = SkipgramTrainer()
    
    # Parse TREC file in chunks
    trec_file_path = r"C:\Eyasu\Thesis_files\DOTGOV\DOTGOV\Full DOTGOV_concatenated\cleaned_concatenated_files.txt"
    print("Parsing TREC file...")
    doc_texts = parse_trec_file_in_chunks(trec_file_path)
    
    # Prepare training data using a generator
    print("Preparing training data...")
    training_data = trainer.prepare_training_data(doc_texts)
    
    # Train model
    print("Training Skip-gram model...")
    model = trainer.train_skipgram_model(training_data)
    
    # Save model
    save_path = os.path.join("..", "Data", "Word_Embedding", "DOTGOV_skipgram_model_cleaned.bin")
    print(f"Saving model to {save_path}...")
    trainer.save_model(model, save_path)
    
    print("Training and saving completed successfully!")

if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dolla\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dolla\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Parsing TREC file...
Preparing training data...
Training Skip-gram model...


TypeError: Using a generator as corpus_iterable can't support 11 passes. Try a re-iterable sequence.

In [ ]:
import os
import re
import string
import logging
from typing import List
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

def process_trec_file(trec_file: str, output_file: str):
    """Process TREC format file"""
    stop_words = set(stopwords.words('english'))
    logging.info("Processing TREC file...")
    
    with open(trec_file, 'r', encoding='utf-8', errors='ignore') as file, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        
        current_doc = []
        processed_docs = 0
        
        for line in file:
            if line.startswith('<DOC>'):
                current_doc = []
            elif line.startswith('</DOC>'):
                if current_doc:
                    text = ' '.join(current_doc)
                    text = re.sub(r'<[^>]+>', '', text)
                    tokens = preprocess_text(text, stop_words)
                    if tokens:
                        outfile.write(' '.join(tokens) + '\n')
                        processed_docs += 1
                        if processed_docs % 1000 == 0:
                            logging.info(f"Processed {processed_docs} documents")
                current_doc = []
            else:
                current_doc.append(line.strip())

def preprocess_text(text: str, stop_words: set) -> List[str]:
    """Preprocess text content"""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return [token for token in tokens if token not in stop_words and token.isalpha() and len(token) > 2]

def train_word2vec(input_file: str, model_path: str):
    """Train Word2Vec model"""
    logging.info("Training Word2Vec model...")
    sentences = LineSentence(input_file)
    model = Word2Vec(
        sentences=sentences,
        vector_size=300,
        window=5,
        min_count=5,
        workers=4,
        sg=0,
        epochs=30
    )
    model.save(f"{model_path}.model")
    model.wv.save_word2vec_format(f"{model_path}.bin", binary=True)

def main():
    trec_file = r"C:\Eyasu\Thesis_files\DOTGOV\DOTGOV\Full DOTGOV_concatenated\concatenated_files.txt"
    processed_file = "processed_trec.txt"
    model_path = "trec_word2vec"
    
    try:
        process_trec_file(trec_file, processed_file)
        train_word2vec(processed_file, model_path)
        logging.info("Training completed successfully!")
    except Exception as e:
        logging.error(f"Error: {str(e)}")
    finally:
        if os.path.exists(processed_file) and input("Remove processed file? (y/n): ").lower() == 'y':
            os.remove(processed_file)

if __name__ == "__main__":
    main()

2025-02-09 11:08:01,959 : INFO : Processing TREC file...
2025-02-09 11:08:04,403 : INFO : Processed 1000 documents
2025-02-09 11:08:07,955 : INFO : Processed 2000 documents
2025-02-09 11:08:11,367 : INFO : Processed 3000 documents
2025-02-09 11:08:14,612 : INFO : Processed 4000 documents
2025-02-09 11:08:19,079 : INFO : Processed 5000 documents
2025-02-09 11:08:23,098 : INFO : Processed 6000 documents
2025-02-09 11:08:26,270 : INFO : Processed 7000 documents
2025-02-09 11:08:29,901 : INFO : Processed 8000 documents
2025-02-09 11:08:33,382 : INFO : Processed 9000 documents
2025-02-09 11:08:36,770 : INFO : Processed 10000 documents
2025-02-09 11:08:40,257 : INFO : Processed 11000 documents
2025-02-09 11:08:43,962 : INFO : Processed 12000 documents
2025-02-09 11:08:46,670 : INFO : Processed 13000 documents
2025-02-09 11:08:49,176 : INFO : Processed 14000 documents
2025-02-09 11:08:52,270 : INFO : Processed 15000 documents
2025-02-09 11:08:55,764 : INFO : Processed 16000 documents
2025-02-

: 

## DOTGOV DATA cleaning

In [3]:
import os
from typing import Iterator, TextIO
import re
from pathlib import Path

def get_output_path(input_path: str) -> str:
    """Generate output file path in the same directory as input file."""
    base_path = Path(input_path)
    return str(base_path.parent / f"cleaned_{base_path.name}")

def process_chunk(chunk: str) -> str:
    """Process a chunk of text by removing TREC tags and cleaning."""
    # Remove all TREC tags
    text = re.sub(r'<[^>]+>', '', chunk)
    
    # Remove empty lines and excessive whitespace
    text = '\n'.join(line.strip() for line in text.split('\n') if line.strip())
    
    return text

def chunk_reader(file_obj: TextIO, chunk_size: int) -> Iterator[str]:
    """Read file in chunks, ensuring we don't split in the middle of a line."""
    remainder = ''
    while True:
        chunk = file_obj.read(chunk_size)
        if not chunk:
            if remainder:
                yield remainder
            break
        
        # Combine with remainder from previous chunk
        chunk = remainder + chunk
        
        # Find last newline in chunk
        last_newline = chunk.rfind('\n')
        if last_newline == -1:
            remainder = chunk
        else:
            yield chunk[:last_newline]
            remainder = chunk[last_newline:]

def clean_trec_file(input_path: str, chunk_size: int = 50*1024*1024) -> None:
    """
    Clean a TREC file by removing tags and unnecessary content.
    Uses streaming to handle large files efficiently.
    
    Args:
        input_path: Path to input TREC file
        chunk_size: Size of chunks to process at once (default 5MB)
    """
    output_path = get_output_path(input_path)
    
    # Try different encodings
    encodings = ['utf-8', 'latin-1', 'ISO-8859-1']
    
    for encoding in encodings:
        try:
            with open(input_path, 'r', encoding=encoding, errors='ignore') as infile, \
                 open(output_path, 'w', encoding='utf-8') as outfile:
                
                print(f"Processing file with encoding: {encoding}")
                print(f"Chunk size: {chunk_size/1024/1024:.1f}MB")
                print(f"Output will be saved to: {output_path}")
                
                # Process file in chunks - using the same chunk_size
                for chunk in chunk_reader(infile, chunk_size):
                    cleaned_text = process_chunk(chunk)
                    if cleaned_text:
                        outfile.write(cleaned_text + '\n')
                
                print("Processing completed successfully!")
                break
                
        except UnicodeDecodeError:
            print(f"Failed with encoding {encoding}, trying next...")
            continue
        except Exception as e:
            print(f"An error occurred: {str(e)}")
            raise

if __name__ == "__main__":
    # Your file path
    trec_file_path = r"C:\Eyasu\Thesis_files\DOTGOV\DOTGOV\Full DOTGOV_concatenated\concatenated_files.txt"
    
    # Set chunk size to 5MB (5 * 1024 * 1024 bytes)
    CHUNK_SIZE = 50 * 1024 * 1024
    
    # Process the file
    clean_trec_file(trec_file_path, chunk_size=CHUNK_SIZE)

Processing file with encoding: utf-8
Chunk size: 50.0MB
Output will be saved to: C:\Eyasu\Thesis_files\DOTGOV\DOTGOV\Full DOTGOV_concatenated\cleaned_concatenated_files.txt
Processing completed successfully!
